# 🎯 DIVINATION V2 - SNIPER ÉLITE AMÉLIORÉ
## Stratégie : Remplacer le maillon faible (LogReg) par le champion (Ultimate Mix)

### Composition Originale (Divination V1) :
1. XGBoost #1 (seed=42) - poids: 0.7
2. XGBoost #2 (seed=2025) - poids: 0.7
3. LightGBM - poids: 1.2 ⭐
4. GradientBoosting - poids: 0.7
5. LogisticRegression - poids: 0.5 ❌ (le plus faible)

### Nouvelle Composition (Divination V2) :
1. XGBoost #1 (seed=42) - poids: 0.7
2. XGBoost #2 (seed=2025) - poids: 0.7
3. LightGBM - poids: 1.2 ⭐
4. GradientBoosting - poids: 0.7
5. **Ultimate Mix (Poisson+LogLoss)** - poids: 1.0 🆕

**Objectif** : Dépasser F1 = 0.77022

In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    VotingClassifier, GradientBoostingClassifier,
    HistGradientBoostingClassifier, HistGradientBoostingRegressor
)
from sklearn.base import BaseEstimator, ClassifierMixin
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("⏳ Chargement des données...")
train_data = pd.read_csv('conversion_data_train.csv')
test_data = pd.read_csv('conversion_data_test.csv')

def feature_engineering(df):
    df_eng = df.copy()
    df_eng['is_active'] = (df_eng['total_pages_visited'] > 2).astype(int)
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    return df_eng

X = feature_engineering(train_data.drop('converted', axis=1))
y = train_data['converted']
X_test = feature_engineering(test_data)

# Encodage pour modèles Gradient Boosting
categorical_cols = ['country', 'source']
X_encoded = X.copy()
X_test_encoded = X_test.copy()

for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]])
    le.fit(combined)
    X_encoded[col] = le.transform(X[col])
    X_test_encoded[col] = le.transform(X_test[col])

print(f"✅ Dataset prêt : {len(X)} lignes\n")

⏳ Chargement des données...
✅ Dataset prêt : 284580 lignes



## 🔧 Création du Wrapper Ultimate Mix

Ultimate Mix doit être intégré dans le VotingClassifier, donc on crée un wrapper compatible sklearn.

In [6]:
class UltimateMixClassifier(BaseEstimator, ClassifierMixin):
    """Wrapper pour Ultimate Mix (Poisson + LogLoss) compatible avec sklearn"""
    
    def __init__(self, learning_rate=0.05, max_iter=500, max_depth=8, l2_regularization=0.1, random_state=42):
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.max_depth = max_depth
        self.l2_regularization = l2_regularization
        self.random_state = random_state
        self.classes_ = np.array([0, 1])
        
    def fit(self, X, y):
        # Identifier les colonnes catégorielles
        cat_cols = ['country', 'source']
        cat_indices = [list(X.columns).index(col) for col in cat_cols if col in X.columns]
        
        # Modèle Poisson
        self.model_poi_ = HistGradientBoostingRegressor(
            loss='poisson',
            learning_rate=self.learning_rate,
            max_iter=self.max_iter,
            max_depth=self.max_depth,
            l2_regularization=self.l2_regularization,
            categorical_features=cat_indices,
            random_state=self.random_state
        )
        self.model_poi_.fit(X, y)
        
        # Modèle LogLoss
        self.model_clf_ = HistGradientBoostingClassifier(
            loss='log_loss',
            learning_rate=self.learning_rate,
            max_iter=self.max_iter,
            max_depth=self.max_depth,
            l2_regularization=self.l2_regularization,
            categorical_features=cat_indices,
            random_state=self.random_state
        )
        self.model_clf_.fit(X, y)
        
        return self
    
    def predict_proba(self, X):
        # Moyenne des prédictions Poisson et LogLoss
        pred_poi = self.model_poi_.predict(X)
        pred_clf = self.model_clf_.predict_proba(X)[:, 1]
        
        avg_pred = (pred_poi + pred_clf) / 2
        
        # Retourner les probabilités pour les deux classes
        proba_class_1 = np.clip(avg_pred, 0, 1)
        proba_class_0 = 1 - proba_class_1
        
        return np.column_stack([proba_class_0, proba_class_1])
    
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

print("✅ Wrapper Ultimate Mix créé")

✅ Wrapper Ultimate Mix créé


## 🎯 DIVINATION V2 - Ensemble Amélioré

In [7]:
print("🧪 DIVINATION V2 - SNIPER ÉLITE AMÉLIORÉ")
print("="*70)

# Preprocessing
numeric_features = ['age', 'total_pages_visited', 'interaction_age_pages', 'pages_per_age']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

# Les 5 modèles (LogReg remplacé par Ultimate Mix)
clf_xgb1 = XGBClassifier(
    n_estimators=350, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, eval_metric='logloss',
    random_state=42, n_jobs=1
)

clf_xgb2 = XGBClassifier(
    n_estimators=350, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, eval_metric='logloss',
    random_state=2025, n_jobs=1
)

clf_lgbm = LGBMClassifier(
    n_estimators=350, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, verbose=-1,
    random_state=42, n_jobs=1
)

clf_gb = GradientBoostingClassifier(
    n_estimators=350, max_depth=4, learning_rate=0.05,
    subsample=0.9, random_state=42
)

# 🆕 Ultimate Mix remplace LogisticRegression
clf_ultimate = UltimateMixClassifier(
    learning_rate=0.05, max_iter=500, max_depth=8,
    l2_regularization=0.1, random_state=42
)

# VotingClassifier avec nouveaux poids
voting_model_v2 = VotingClassifier(
    estimators=[
        ('xgb1', clf_xgb1),
        ('xgb2', clf_xgb2),
        ('lgbm', clf_lgbm),
        ('gb', clf_gb),
        ('ultimate_mix', clf_ultimate)  # 🆕 Remplace log_recall
    ],
    voting='soft',
    weights=[0.7, 0.7, 1.2, 0.7, 1.0],  # Ultimate Mix a un poids de 1.0
    n_jobs=-1
)

pipeline_v2 = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', voting_model_v2)
])

print("📊 Configuration :")
print("   - XGBoost #1 (seed=42)     : poids 0.7")
print("   - XGBoost #2 (seed=2025)   : poids 0.7")
print("   - LightGBM                 : poids 1.2 ⭐")
print("   - GradientBoosting         : poids 0.7")
print("   - Ultimate Mix (Poi+Log)   : poids 1.0 🆕")
print("\n⏳ Entraînement en 10-Fold CV (peut prendre 3-4 minutes)...\n")

🧪 DIVINATION V2 - SNIPER ÉLITE AMÉLIORÉ
📊 Configuration :
   - XGBoost #1 (seed=42)     : poids 0.7
   - XGBoost #2 (seed=2025)   : poids 0.7
   - LightGBM                 : poids 1.2 ⭐
   - GradientBoosting         : poids 0.7
   - Ultimate Mix (Poi+Log)   : poids 1.0 🆕

⏳ Entraînement en 10-Fold CV (peut prendre 3-4 minutes)...



In [8]:
# Cross-validation manuelle pour avoir le contrôle total
n_folds = 10
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"   Fold {fold}/{n_folds}...", end=" ")
    
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_val_fold = y.iloc[val_idx]
    
    # Entraînement
    pipeline_v2.fit(X_train_fold, y_train_fold)
    
    # Prédictions
    val_proba = pipeline_v2.predict_proba(X_val_fold)[:, 1]
    oof_predictions[val_idx] = val_proba
    
    # Score du fold
    fold_f1 = f1_score(y_val_fold, (val_proba >= 0.5).astype(int))
    fold_scores.append(fold_f1)
    print(f"F1: {fold_f1:.5f}")

print(f"\n✅ Cross-validation terminée")
print(f"   F1 moyen par fold : {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")

   Fold 1/10... 

ValueError: The estimator UltimateMixClassifier should be a classifier.

In [ ]:
# Optimisation du seuil
print("\n🎯 Optimisation du seuil...")

best_f1 = 0
best_threshold = 0

for threshold in np.linspace(oof_predictions.min(), oof_predictions.max(), 1000):
    preds = (oof_predictions >= threshold).astype(int)
    score = f1_score(y, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

# Prédictions finales avec seuil optimal
final_predictions = (oof_predictions >= best_threshold).astype(int)

# Métriques complètes
metrics_v2 = {
    'f1': best_f1,
    'roc_auc': roc_auc_score(y, oof_predictions),
    'precision': precision_score(y, final_predictions),
    'recall': recall_score(y, final_predictions),
    'accuracy': accuracy_score(y, final_predictions),
    'threshold': best_threshold
}

print(f"   Seuil optimal : {best_threshold:.6f}")
print(f"\n{'='*70}")
print("🏆 RÉSULTATS DIVINATION V2")
print(f"{'='*70}")
print(f"   F1-Score  : {metrics_v2['f1']:.5f}")
print(f"   ROC-AUC   : {metrics_v2['roc_auc']:.5f}")
print(f"   Precision : {metrics_v2['precision']:.5f}")
print(f"   Recall    : {metrics_v2['recall']:.5f}")
print(f"   Accuracy  : {metrics_v2['accuracy']:.5f}")
print(f"{'='*70}")

## 📊 Comparaison V1 vs V2

In [ ]:
# Résultats V1 (depuis benchmark_ultimate.ipynb)
metrics_v1 = {
    'f1': 0.77022,
    'roc_auc': 0.98577,
    'precision': 0.81007,
    'recall': 0.73410,
    'accuracy': 0.98587
}

print("\n" + "="*80)
print("📊 COMPARAISON DIVINATION V1 vs V2")
print("="*80)
print(f"\n{'Métrique':<15} {'V1 (Original)':<20} {'V2 (Ultimate Mix)':<20} {'Δ':<15}")
print("-"*80)

for metric in ['f1', 'roc_auc', 'precision', 'recall', 'accuracy']:
    v1_val = metrics_v1[metric]
    v2_val = metrics_v2[metric]
    delta = v2_val - v1_val
    delta_str = f"{delta:+.5f}" if delta != 0 else "="
    emoji = "📈" if delta > 0 else "📉" if delta < 0 else "➡️"
    
    print(f"{metric.upper():<15} {v1_val:<20.5f} {v2_val:<20.5f} {emoji} {delta_str}")

print("\n" + "="*80)

# Verdict
if metrics_v2['f1'] > metrics_v1['f1']:
    improvement = (metrics_v2['f1'] - metrics_v1['f1']) * 100
    print(f"\n🎉 VICTOIRE ! Divination V2 améliore le F1-Score de {improvement:.3f} points !")
    print(f"   Nouveau champion : F1 = {metrics_v2['f1']:.5f}")
elif metrics_v2['f1'] == metrics_v1['f1']:
    print(f"\n➡️ ÉGALITÉ : Les deux versions ont le même F1-Score ({metrics_v2['f1']:.5f})")
else:
    decline = (metrics_v1['f1'] - metrics_v2['f1']) * 100
    print(f"\n⚠️ ATTENTION : V2 est légèrement inférieur de {decline:.3f} points")
    print(f"   V1 reste champion : F1 = {metrics_v1['f1']:.5f}")

print("\n" + "="*80)

## 🚀 Entraînement Final et Génération des Prédictions

In [ ]:
print("\n🚀 Entraînement du modèle final sur tout le dataset...\n")

# Entraînement sur toutes les données
pipeline_v2.fit(X, y)

# Prédictions sur le test set
test_proba = pipeline_v2.predict_proba(X_test)[:, 1]
test_predictions = (test_proba >= best_threshold).astype(int)

# Création du fichier de soumission
submission = pd.DataFrame({
    'converted': test_predictions
})

submission.to_csv('divination_v2_predictions.csv', index=False)

print("✅ Prédictions générées : divination_v2_predictions.csv")
print(f"   Taux de conversion prédit : {test_predictions.mean():.4%}")
print(f"   Nombre de conversions : {test_predictions.sum()} / {len(test_predictions)}")
print(f"\n🎯 Seuil utilisé : {best_threshold:.6f}")
print(f"\n{'='*70}")
print("🏆 DIVINATION V2 - PRÊT POUR SOUMISSION")
print(f"{'='*70}")

## 📝 Conclusion

### Changements apportés :
- ❌ **Retiré** : LogisticRegression (poids 0.5)
- ✅ **Ajouté** : Ultimate Mix - Poisson + LogLoss (poids 1.0)

### Avantages de V2 :
1. **Diversité accrue** : Combine approches régression (Poisson) et classification
2. **Performance prouvée** : Ultimate Mix a un F1 de 0.76877 (vs LogReg beaucoup plus faible)
3. **Complémentarité** : Apporte une vision différente des autres modèles

### Fichiers générés :
- `divination_v2_predictions.csv` : Prédictions pour soumission Kaggle